In [1]:
import sqlite3

connection = sqlite3.connect("Sales_Intelligence_Hub_Database.db")
cursor = connection.cursor()

cursor.execute
results = cursor.fetchall()


In [13]:
# 1. Define the SQL statement based on your requirements
create_table_query = """
CREATE TABLE IF NOT EXISTS branches (
    branch_id INT AUTO_INCREMENT PRIMARY KEY,
    branch_name VARCHAR(100) NOT NULL,
    branch_admin_name VARCHAR(100) NOT NULL
);
"""

try:
    # 2. Execute the query using your existing cursor
    cursor.execute(create_table_query)
    
    # 3. Commit the changes to save the table structure
    connection.commit()
    print("Table 'branches' created successfully!")

except Exception as e:
    print(f"Failed to create table: {e}")


Table 'branches' created successfully!


In [ ]:
create_table_query_1 = """
CREATE TABLE IF NOT EXISTS customer_sales_table(
    sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
    branch_id INT,
    date DATE,
    name VARCHAR(100),
    mobile_number VARCHAR(15),
    product_name VARCHAR(100),
    gross_sales DECIMAL(12,2),
    received_amount DECIMAL(12,2),
    status TEXT CHECK(status IN ('Open', 'Close')),
    FOREIGN KEY (branch_id) REFERENCES branches(branch_id)
);
"""

cursor.execute(create_table_query_1)


create_table_query_2 = """
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username VARCHAR(100),
    password VARCHAR(255),
    branch_id INTEGER,
    role TEXT CHECK(role IN ('Super Admin', 'Admin')),
    email VARCHAR(255) UNIQUE,
    FOREIGN KEY (branch_id) REFERENCES branches(branch_id)
);
"""

cursor.execute(create_table_query_2)

create_table_query_3 = """
CREATE TABLE IF NOT EXISTS payment_splits (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    sale_id INTEGER,
    payment_date DATE,
    amount_paid DECIMAL(12,2),
    payment_method VARCHAR(50),
    FOREIGN KEY (sale_id) REFERENCES customer_sales_table(sale_id)
);
"""

cursor.execute(create_table_query_3)

In [22]:
import pandas as pd

df_1 = pd.read_csv("branches.csv")
df_2 = pd.read_csv("customer_sales.csv") 
df_3 = pd.read_csv("users.csv")
df_4 = pd.read_csv("payment_splits.csv")

In [25]:
df_1.to_sql('branches', connection, if_exists='append', index=False)
df_2.to_sql('customer_sales_table', connection, if_exists='append', index=False)
df_3.to_sql('users', connection, if_exists='append', index=False)
df_4.to_sql('payment_splits', connection, if_exists='append', index=False)

2011

In [5]:
cursor.execute("ALTER TABLE customer_sales_table ADD COLUMN pending_amount DECIMAL(12,2);")
connection.commit()

In [ ]:
cursor.execute("PRAGMA foreign_keys = ON;")  # Enable foreign key support

In [10]:
# Creating a trigger to update received_amount and pending_amount in customer_sales_table after inserting into payment_splits
trg_pending_amt = """
CREATE TRIGGER IF NOT EXISTS trg_pending_amt
AFTER INSERT ON payment_splits
BEGIN
    UPDATE customer_sales_table
    SET received_amount = (
            SELECT COALESCE(SUM(amount_paid), 0)
            FROM payment_splits
            WHERE sale_id = NEW.sale_id
        ),
        pending_amount = gross_sales - (
            SELECT COALESCE(SUM(amount_paid), 0)
            FROM payment_splits
            WHERE sale_id = NEW.sale_id
        )
    WHERE sale_id = NEW.sale_id;
END;
"""

cursor.execute(trg_pending_amt)
connection.commit()

In [12]:
# Trigger: fills pending_amount the instant a new sale is created
trg_init_pending = """
CREATE TRIGGER IF NOT EXISTS trg_init_pending
AFTER INSERT ON customer_sales_table
BEGIN
    UPDATE customer_sales_table
    SET pending_amount = gross_sales - received_amount
    WHERE sale_id = NEW.sale_id;
END;
"""

cursor.execute(trg_init_pending)
connection.commit()

In [13]:
# Trigger: auto-closes the sale once pending_amount reaches 0
trg_close_sale = """
CREATE TRIGGER IF NOT EXISTS trg_close_sale
AFTER UPDATE OF received_amount ON customer_sales_table
WHEN NEW.pending_amount <= 0 AND NEW.status = 'Open'
BEGIN
    UPDATE customer_sales_table
    SET status = 'Close'
    WHERE sale_id = NEW.sale_id;
END;
"""

cursor.execute(trg_close_sale)
connection.commit()

In [14]:
# Trigger: rejects any new payment_splits insert if the sale is already Closed
trg_reject_closed = """
CREATE TRIGGER IF NOT EXISTS trg_reject_closed
BEFORE INSERT ON payment_splits
WHEN (SELECT status FROM customer_sales_table WHERE sale_id = NEW.sale_id) = 'Close'
BEGIN
    SELECT RAISE(ABORT, 'Cannot add payment: sale is already closed');
END;
"""

cursor.execute(trg_reject_closed)
connection.commit()

In [15]:
# One-time backfill: your existing 1000 rows were loaded before trg_init_pending
# existed, so they still have pending_amount = NULL — this fixes them once
cursor.execute("""
UPDATE customer_sales_table
SET pending_amount = gross_sales - received_amount
WHERE pending_amount IS NULL;
""")
connection.commit()

In [3]:
import pandas as pd

def authenticate(username, password, connection):
    query = "SELECT user_id, username, role, branch_id FROM users WHERE username = ? AND password = ?;"
    result = pd.read_sql(query, connection, params=(username, password))
    if result.empty:
        return None  # login failed
    return result.iloc[0]  # a row with user_id, username, role, branch_id


def get_sales(connection, role, branch_id):
    if role == "Super Admin":
        query = "SELECT * FROM customer_sales_table;"
        return pd.read_sql(query, connection)
    else:  # Admin
        query = "SELECT * FROM customer_sales_table WHERE branch_id = ?;"
        return pd.read_sql(query, connection, params=(branch_id,))

In [4]:
print(authenticate("admin_chennai", "admin123", connection))
print(authenticate("superadmin", "super123", connection))
print(authenticate("admin_chennai", "wrongpassword", connection))

df_all = get_sales(connection, "Super Admin", None)
print(len(df_all))

df_branch1 = get_sales(connection, "Admin", 1)
print(df_branch1['branch_id'].unique())

user_id                  2
username     admin_chennai
role                 Admin
branch_id                1
Name: 0, dtype: object
user_id                1
username      superadmin
role         Super Admin
branch_id           None
Name: 0, dtype: object
None
1000
[1]


In [ ]:
pip install streamlit  #install streamlit package

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
